# SAÉ — Migration vers NoSQL

## Contexte et contraintes métier

La base décrit la programmation d’une radio et ses diffusions. Les requêtes s’appuient notamment sur les règles suivantes :

- **Protection du jeune public** : aucun titre `explicit` ne doit être diffusé pendant les tranches protégées du matin et de l’après-midi.
- **Quota de nouveautés** : l’antenne doit diffuser un minimum de titres récent·e·s, sélectionné·e·s selon la date de sortie de leur album.
- **Déclaration des diffusions** : pour les sociétés de gestion de droits, il faut pouvoir obtenir rapidement, pour chaque artiste, le nombre de diffusions et la durée cumulée correspondante.

Pour chaque requête, choisissez un ordre de présentation lorsqu’il n’est pas évident, afin de pouvoir comparer les résultats après migration.

In [ ]:
!pip install pandas --quiet

In [ ]:
import sqlite3
import pandas as pd
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# Connection to Radio Kiosque database
sqlite_connection = sqlite3.connect("data/RadioKiosque.sqlite")

Comment requêter la base de données ?

In [ ]:
pd.read_sql_query(
    """
        SELECT host, title
        FROM Shows
        ORDER BY host
    """,
    con=sqlite_connection
)

In [ ]:
#Première analyse pour découvrir la base de données

#Nombre de lignes dans quelques tables 

pd.read_sql_query(
    """
        SELECT 
        (SELECT COUNT(*) FROM Shows) AS "Nombre de Shows",
        (SELECT COUNT(*) FROM Broadcasts) AS "Nombre de broadcasts",
        (SELECT COUNT(*) FROM listeners) AS "Nombre d'auditeurs",
        (SELECT COUNT(*) FROM requests) AS "Nombre de requêtes",
        (SELECT COUNT(*) FROM Tracks) AS "Nombre de chansons"
    """,
    con=sqlite_connection
)

In [ ]:
#Afficher quelques lignes de chansons explicites

pd.read_sql_query(
    """
        SELECT name as "Titre", explicit
        FROM Tracks
        WHERE explicit = 1
        LIMIT 10
    """,
    con=sqlite_connection
)

### 1. Auditeur·ices sans demande

Lister les auditeur·ices n’ayant jamais effectué de demande musicale.

In [ ]:
Q1 = pd.read_sql_query(
    """
        SELECT Listeners.pseudo AS "Pseudonyme"
        FROM Listeners
        LEFT JOIN Requests ON Listeners.listener_id = Requests.listener_id
        WHERE Requests.Listener_id IS NULL
        ORDER BY Listeners.listener_id
    """,
    con=sqlite_connection
)

Q1.to_csv("references/q1.csv", index = False)

Q1

### 2. Bilan par animateur·ice

Pour chaque animateur·ice, indiquer le nombre d’émissions, le nombre de diffusions et la durée totale diffusée.

In [ ]:
Q2 = pd.read_sql_query(
    """
        SELECT Shows.host as "Animateur(trice)", 
            COUNT( DISTINCT Shows.show_id) as "Nombre d'émissions", 
            COUNT(Broadcasts.broadcast_id) as "Nombre de diffusions",
            SUM(Tracks.duration_ms) as "Durée totale diffusée"
        FROM Shows
        LEFT JOIN Broadcasts ON Shows.show_id = Broadcasts.show_id
        LEFT JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        GROUP BY Shows.host
        ORDER BY Shows.host
    """,
    con=sqlite_connection
)

Q2.to_csv("references/q2.csv", index = False)

Q2

### 3. Bilan par tranche horaire

Pour chaque tranche horaire, indiquer le nombre de diffusions et le nombre de titres explicites diffusés.

In [ ]:
Q3 = pd.read_sql_query(
    """
        SELECT Shows.time_slot as "Tranche horaire",
            COUNT(Broadcasts.broadcast_id) AS "Nombre de diffusions", 
            SUM(Tracks.explicit) AS "Nombre de titres explicites"
        FROM Shows
        LEFT JOIN Broadcasts ON Shows.show_id = Broadcasts.show_id
        LEFT JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        GROUP BY Shows.time_slot
        Order By Shows.time_slot
    """,
    con=sqlite_connection
)

Q3.to_csv("references/q3.csv", index = False)

Q3

### 4. Bilan par titre

Pour chaque titre, indiquer le nombre de diffusions, l’audience cumulée et le nombre d’émissions différentes où il a été programmé.

In [ ]:
Q4 = pd.read_sql_query(
    """
        SELECT Shows.time_slot as "Tranche horaire",
            COUNT(Broadcasts.broadcast_id) AS "Nombre de diffusions", 
            SUM(Tracks.explicit) AS "Nombre de titres explicites"
        FROM Shows
        LEFT JOIN Broadcasts ON Shows.show_id = Broadcasts.show_id
        LEFT JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        GROUP BY Shows.time_slot
        Order By Shows.time_slot
    """,
    con=sqlite_connection
)

Q4.to_csv("references/q4.csv", index = False)

Q4

### 5. Bilan par artiste

Pour chaque artiste (principal·e ou invité·e), indiquer le nombre de titres au catalogue, le nombre de diffusions et la durée cumulée diffusée. Inclure les artistes jamais diffusé·es. Trier par ordre décroissant du nombre de diffusions.

In [ ]:
pd.read_sql_query(
    """
        SELECT Listeners.pseudo AS "Pseudonyme"
        FROM Listeners
        LEFT JOIN Requests ON Listeners.listener_id = Requests.listener_id
        WHERE Requests.Listener_id IS NULL
        ORDER BY Listeners.listener_id
    """,
    con=sqlite_connection
)

### 6. Diffusions par décennie et tranche horaire

Présenter le nombre de diffusions sous forme de tableau croisé entre la décennie de sortie de l’album et la tranche horaire.

Voici le format attendu :

| decade | morning | afternoon | evening | night |
|-------:|--------:|----------:|--------:|------:|
|   1960 |         |           |         |       |
|   1970 |         |           |         |       |
|   1980 |         |           |         |       |
|   1990 |         |           |         |       |
|   2000 |         |           |         |       |
|   2010 |         |           |         |       |
|   2020 |         |           |         |       |

&#128161; Exécuter la requête en SQL avec la décennie, la tranche horaire et le nombre de diffusion. Puis pivoter la table avec la fonction `pivot_table` de Pandas.

In [ ]:
pd.read_sql_query(
    """
        SELECT Listeners.pseudo AS "Pseudonyme"
        FROM Listeners
        LEFT JOIN Requests ON Listeners.listener_id = Requests.listener_id
        WHERE Requests.Listener_id IS NULL
        ORDER BY Listeners.listener_id
    """,
    con=sqlite_connection
)

### 7. Audience par décennie et tranche horaire

Présenter l’audience cumulée sous forme de tableau croisé entre la décennie de sortie de l’album et la tranche horaire.

In [ ]:
Q7 = pd.read_sql_query(
    """
        SELECT
            CAST(substr(Albums.release_date, 1, 3) AS INTEGER) * 10 AS decade,
            SUM(CASE WHEN Shows.time_slot = 'morning'   THEN Broadcasts.listeners ELSE 0 END) AS morning,
            SUM(CASE WHEN Shows.time_slot = 'afternoon' THEN Broadcasts.listeners ELSE 0 END) AS afternoon,
            SUM(CASE WHEN Shows.time_slot = 'evening'   THEN Broadcasts.listeners ELSE 0 END) AS evening,
            SUM(CASE WHEN Shows.time_slot = 'night'     THEN Broadcasts.listeners ELSE 0 END) AS night
        FROM Broadcasts
        JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        JOIN Albums ON Tracks.album_id = Albums.album_id
        JOIN Shows ON Broadcasts.show_id = Shows.show_id
        GROUP BY decade
        ORDER BY decade
    """,
    con=sqlite_connection
)

#Q7.to_csv("references/q7.csv", index = False)

Q7

### 8. Top 10 des audiences moyennes

Parmi les titres diffusés au moins trois fois, lister les dix titres ayant la plus forte audience moyenne par diffusion. Trier par l'audience moyenne.

In [ ]:
Q8 = pd.read_sql_query(
    """
        SELECT Tracks.track_id, Tracks.name, COUNT(*) AS number_broadcasts, AVG(Broadcasts.listeners) AS average_audience
        FROM Broadcasts
        JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        GROUP BY Tracks.track_id, Tracks.name
        HAVING COUNT(*) >= 3
        ORDER BY average_audience DESC
        LIMIT 10
    """,
    con=sqlite_connection
)

Q8.to_csv("references/q8.csv", index = False)

Q8

### 9. Infractions à la protection du jeune public

Lister chaque diffusion d’un titre explicite dans une émission protégée, avec le titre, l’identifiant de l’émission et l’horodatage.

In [ ]:
Q9 = pd.read_sql_query(
    """
        SELECT Tracks.name AS track_title, Broadcasts.show_id, Broadcasts.played_at
        FROM Broadcasts
        JOIN Tracks ON Broadcasts.track_id = Tracks.track_id
        JOIN Shows ON Broadcasts.show_id = Shows.show_id
        WHERE Tracks.explicit = 1 AND Shows.is_protected = 1
        ORDER BY Broadcasts.played_at
    """,
    con=sqlite_connection
)

#Q9.to_csv("references/q9.csv", index = False)

Q9

### 10. Albums diffusés en entier

Lister les albums dont tous les titres ont été diffusés au moins une fois.

In [ ]:
Q10 = pd.read_sql_query(
    """
        SELECT Albums.album_id, Albums.name
        FROM Albums
        JOIN Tracks ON Tracks.album_id = Albums.album_id
        LEFT JOIN Broadcasts ON Broadcasts.track_id = Tracks.track_id
        GROUP BY Albums.album_id, Albums.name
        HAVING COUNT(DISTINCT Tracks.track_id) = COUNT(DISTINCT Broadcasts.track_id)
        ORDER BY Albums.name
    """,
    con=sqlite_connection
)

#Q10.to_csv("references/q10.csv", index = False)

Q10

### 11. Podcasts classés par pays

Pour chaque pays, indiquer le nombre de podcasts classés, le nombre de podcasts explicites et le nombre de podcasts dont la fiche est fiable.

In [ ]:
Q11 = pd.read_sql_query(
    """
        SELECT PodcastRankings.country, COUNT(*) AS nb_podcasts, SUM(Podcasts.explicit) AS nb_explicit, SUM(Podcasts.match_status = 'exact') AS nb_exact_match
        FROM PodcastRankings
        JOIN Podcasts ON PodcastRankings.podcast_id = Podcasts.podcast_id
        GROUP BY PodcastRankings.country
        ORDER BY PodcastRankings.country
    """,
    con=sqlite_connection
)

#Q11.to_csv("references/q11.csv", index = False)

Q11

### 12. Podcasts programmables

Lister les podcasts programmables, avec leur nom, leur éditeur, le nombre de pays dans le top 10 desquels ils figurent et leur meilleur rang. Trier du plus présent au moins présent, puis par meilleur rang.

In [ ]:
Q12 = pd.read_sql_query(
    """
        SELECT Podcasts.podcast_id, Podcasts.name AS podcast_name, Podcasts.publisher, COUNT(*) AS nb_countries, MIN(PodcastRankings.rank) AS best_rank
        FROM Podcasts
        JOIN PodcastRankings ON PodcastRankings.podcast_id = Podcasts.podcast_id
        WHERE Podcasts.match_status = 'exact' AND Podcasts.explicit = 0 AND PodcastRankings.rank <= 10
        GROUP BY Podcasts.podcast_id, Podcasts.name, Podcasts.publisher
        ORDER BY nb_countries DESC, best_rank ASC
    """,
    con=sqlite_connection
)

#Q12.to_csv("references/q12.csv", index = False)

Q12

### 13. Durée et activité des podcasts programmables

Pour chaque podcast programmable, indiquer le nombre d’épisodes connus, la durée moyenne de ses épisodes (en minutes) et la date de son dernier épisode. Ne garder que les podcasts dont la durée moyenne ne dépasse pas une heure.

In [ ]:
Q13 = pd.read_sql_query(
    """
        SELECT Podcasts.podcast_id, Podcasts.name AS podcast_name, COUNT(*) AS nb_episodes, ROUND(AVG(PodcastEpisodes.duration_ms) / 60000.0, 1) AS avg_minutes, MAX(PodcastEpisodes.release_date) AS last_release
        FROM Podcasts
        JOIN PodcastEpisodes ON PodcastEpisodes.podcast_id = Podcasts.podcast_id
        WHERE Podcasts.match_status = 'exact' AND Podcasts.explicit = 0
        GROUP BY Podcasts.podcast_id, Podcasts.name
        HAVING AVG(PodcastEpisodes.duration_ms) / 60000.0 <= 60
        ORDER BY Podcasts.podcast_id
    """,
    con=sqlite_connection
)

#Q13.to_csv("references/q13.csv", index = False)

Q13

In [ ]:
# Close the connection to the database
sqlite_connection.close()